# Parker Hannifin Component Predictive Maintenance — PowerPoint Generator

Generates a professional executive-ready `.pptx` presentation using real data from the Parker Hannifin aerospace component predictive maintenance platform. Powered by `python-pptx`.

In [0]:
%pip install python-pptx --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
import os

# ── Color palette ──
DARK_BLUE = RGBColor(0x1B, 0x3A, 0x5C)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
ACCENT_ORANGE = RGBColor(0xFF, 0x6B, 0x35)
DBX_RED = RGBColor(0xFF, 0x36, 0x21)
LIGHT_GRAY = RGBColor(0xF2, 0xF2, 0xF2)
MED_GRAY = RGBColor(0xE0, 0xE0, 0xE0)
DARK_TEXT = RGBColor(0x2D, 0x2D, 0x2D)
SUBTITLE_BLUE = RGBColor(0x4A, 0x7C, 0xA8)
CRITICAL_RED = RGBColor(0xD9, 0x32, 0x25)
HIGH_AMBER = RGBColor(0xE8, 0x8C, 0x00)
MEDIUM_YELLOW = RGBColor(0xCC, 0xA3, 0x00)
LOW_GREEN = RGBColor(0x27, 0xAE, 0x60)

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

SLIDE_W = prs.slide_width
SLIDE_H = prs.slide_height

# ── Helper functions ──
def add_background(slide, color=DARK_BLUE):
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = color

def add_bottom_bar(slide):
    bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, SLIDE_H - Inches(0.08), SLIDE_W, Inches(0.08))
    bar.fill.solid()
    bar.fill.fore_color.rgb = DBX_RED
    bar.line.fill.background()

def add_accent_line(slide, left, top, width, color=ACCENT_ORANGE, height=Inches(0.04)):
    line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
    line.fill.solid()
    line.fill.fore_color.rgb = color
    line.line.fill.background()

def add_slide_number(slide, num):
    txBox = slide.shapes.add_textbox(SLIDE_W - Inches(1), SLIDE_H - Inches(0.5), Inches(0.8), Inches(0.35))
    tf = txBox.text_frame
    p = tf.paragraphs[0]
    p.text = str(num)
    p.font.size = Pt(11)
    p.font.color.rgb = RGBColor(0x99, 0x99, 0x99)
    p.alignment = PP_ALIGN.RIGHT

def set_cell_text(cell, text, font_size=14, bold=False, color=DARK_TEXT, alignment=PP_ALIGN.LEFT):
    cell.text = ""
    p = cell.text_frame.paragraphs[0]
    p.text = str(text)
    p.font.size = Pt(font_size)
    p.font.bold = bold
    p.font.color.rgb = color
    p.alignment = alignment
    cell.text_frame.word_wrap = True
    cell.vertical_anchor = MSO_ANCHOR.MIDDLE

def add_metric_box(slide, left, top, width, height, value, label, value_color=ACCENT_ORANGE):
    box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left, top, width, height)
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(0x0F, 0x25, 0x3D)
    box.line.color.rgb = RGBColor(0x2A, 0x5A, 0x8C)
    box.line.width = Pt(1)
    tf = box.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.CENTER
    run = p.add_run()
    run.text = str(value)
    run.font.size = Pt(36)
    run.font.bold = True
    run.font.color.rgb = value_color
    p2 = tf.add_paragraph()
    p2.alignment = PP_ALIGN.CENTER
    run2 = p2.add_run()
    run2.text = label
    run2.font.size = Pt(14)
    run2.font.color.rgb = RGBColor(0xBB, 0xBB, 0xBB)

def add_bullet_slide_content(slide, bullets, left, top, width, height, font_size=20, color=WHITE):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = True
    for i, bullet in enumerate(bullets):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_after = Pt(12)
        p.level = 0
        run_bullet = p.add_run()
        run_bullet.text = "\u25B8  "
        run_bullet.font.size = Pt(font_size)
        run_bullet.font.color.rgb = ACCENT_ORANGE
        run_bullet.font.bold = True
        run_text = p.add_run()
        run_text.text = bullet
        run_text.font.size = Pt(font_size)
        run_text.font.color.rgb = color

def add_section_title(slide, title, subtitle=None):
    # Title
    txBox = slide.shapes.add_textbox(Inches(0.8), Inches(0.5), Inches(11), Inches(0.8))
    tf = txBox.text_frame
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(36)
    p.font.bold = True
    p.font.color.rgb = WHITE
    # Accent line under title
    add_accent_line(slide, Inches(0.8), Inches(1.25), Inches(3))
    if subtitle:
        txBox2 = slide.shapes.add_textbox(Inches(0.8), Inches(1.45), Inches(11), Inches(0.5))
        tf2 = txBox2.text_frame
        p2 = tf2.paragraphs[0]
        p2.text = subtitle
        p2.font.size = Pt(18)
        p2.font.color.rgb = SUBTITLE_BLUE


# ═══════════════════════════════════════════════════
# SLIDE 1 — Title
# ═══════════════════════════════════════════════════
slide1 = prs.slides.add_slide(prs.slide_layouts[6])  # blank
add_background(slide1)

# Top accent bar
top_bar = slide1.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, SLIDE_W, Inches(0.06))
top_bar.fill.solid()
top_bar.fill.fore_color.rgb = ACCENT_ORANGE
top_bar.line.fill.background()

# Side accent
side = slide1.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, Inches(2.5), Inches(0.08), Inches(2.5))
side.fill.solid()
side.fill.fore_color.rgb = ACCENT_ORANGE
side.line.fill.background()

# Title
txBox = slide1.shapes.add_textbox(Inches(1.2), Inches(2.0), Inches(10), Inches(1.5))
tf = txBox.text_frame
p = tf.paragraphs[0]
run = p.add_run()
run.text = "Parker Hannifin Aerospace"
run.font.size = Pt(48)

# Subtitle line 2
p_sub = tf.add_paragraph()
run_sub = p_sub.add_run()
run_sub.text = "Component Predictive Maintenance"
run_sub.font.size = Pt(44)
run_sub.font.bold = True
run_sub.font.color.rgb = ACCENT_ORANGE
run.font.bold = True
run.font.color.rgb = WHITE

add_accent_line(slide1, Inches(1.2), Inches(3.5), Inches(4), ACCENT_ORANGE, Inches(0.05))

# Subtitle
txBox2 = slide1.shapes.add_textbox(Inches(1.2), Inches(3.8), Inches(10), Inches(0.8))
tf2 = txBox2.text_frame
p2 = tf2.paragraphs[0]
run2 = p2.add_run()
run2.text = "Proactive Part Failure Prevention — Powered by Databricks"
run2.font.size = Pt(26)
run2.font.color.rgb = SUBTITLE_BLUE

# Databricks tag
txBox3 = slide1.shapes.add_textbox(Inches(1.2), Inches(5.5), Inches(5), Inches(0.4))
tf3 = txBox3.text_frame
p3 = tf3.paragraphs[0]
run3 = p3.add_run()
run3.text = "PARKER HANNIFIN  |  DATABRICKS  |  CONFIDENTIAL"
run3.font.size = Pt(12)
run3.font.color.rgb = RGBColor(0x66, 0x88, 0xAA)
run3.font.letter_spacing = Pt(3)

add_bottom_bar(slide1)
add_slide_number(slide1, 1)


# ═══════════════════════════════════════════════════
# SLIDE 2 — The Challenge
# ═══════════════════════════════════════════════════
slide2 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide2)
add_section_title(slide2, "The Challenge")

bullets2 = [
    "Parker aerospace components not receiving proper maintenance → flight delays, AOG liability, warranty claims",
    "Airlines underreporting maintenance anomalies: 9.8% of service records have embedded issues",
    "$8.5M warranty exposure across 1,640 installed components — Parker needs proactive visibility",
]
add_bullet_slide_content(slide2, bullets2, Inches(0.8), Inches(2.0), Inches(11), Inches(4), font_size=22)

# Impact stat boxes
add_metric_box(slide2, Inches(1.0), Inches(4.8), Inches(3.2), Inches(1.5), "$8.5M", "Total warranty exposure across fleet", CRITICAL_RED)
add_metric_box(slide2, Inches(5.0), Inches(4.8), Inches(3.2), Inches(1.5), "9.8%", "Maintenance records with anomalies", HIGH_AMBER)
add_metric_box(slide2, Inches(9.0), Inches(4.8), Inches(3.2), Inches(1.5), "969", "Anomaly events detected")

add_bottom_bar(slide2)
add_slide_number(slide2, 2)


# ═══════════════════════════════════════════════════
# SLIDE 3 — Fleet Overview
# ═══════════════════════════════════════════════════
slide3 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide3)
add_section_title(slide3, "Component Portfolio", "Real-time monitoring across 1,640 installed Parker components")

metrics = [
    ("1,640", "Parker Components", ACCENT_ORANGE),
    ("328", "Aircraft Monitored", ACCENT_ORANGE),
    ("280", "Critical Risk", CRITICAL_RED),
    ("$8.5M", "Warranty Exposure", CRITICAL_RED),
    ("10", "Airline Customers", ACCENT_ORANGE),
]
start_x = Inches(0.5)
for i, (val, label, clr) in enumerate(metrics):
    add_metric_box(slide3, start_x + Inches(2.45 * i), Inches(2.2), Inches(2.2), Inches(1.8), val, label, clr)

# Additional context text
txBox = slide3.shapes.add_textbox(Inches(0.8), Inches(4.8), Inches(11), Inches(1.5))
tf = txBox.text_frame
tf.word_wrap = True
p = tf.paragraphs[0]
p.space_after = Pt(8)
run = p.add_run()
run.text = "5 Parker component types: Hydraulic Actuators ($28.5K), Fuel Control Valves ($14.2K), "
run.font.size = Pt(16)
run.font.color.rgb = RGBColor(0xBB, 0xBB, 0xBB)
run2 = p.add_run()
run2.text = "ECS Controllers ($19.7K), Pneumatic Valves ($8.9K), Avionics Sensors ($5.4K) — monitored via Databricks Lakehouse."
run2.font.size = Pt(16)
run2.font.color.rgb = RGBColor(0xBB, 0xBB, 0xBB)

add_bottom_bar(slide3)
add_slide_number(slide3, 3)


# ═══════════════════════════════════════════════════
# SLIDE 4 — Anomaly Detection & Analysis
# ═══════════════════════════════════════════════════
slide4 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide4)
add_section_title(slide4, "Maintenance Anomaly Detection", "6 anomaly types detected across 9,839 service records")

# Left side — Top anomalies
txLabel = slide4.shapes.add_textbox(Inches(0.8), Inches(2.0), Inches(5), Inches(0.4))
tfL = txLabel.text_frame
pL = tfL.paragraphs[0]
pL.text = "Anomaly Types by Frequency"
pL.font.size = Pt(18)
pL.font.bold = True
pL.font.color.rgb = ACCENT_ORANGE

anomalies = [("MISSED_MAINTENANCE_INTERVAL", 217), ("PREMATURE_WEAR_INDICATOR", 200), ("UNDOCUMENTED_REPLACEMENT", 174),
             ("FALSE_COMPLIANCE_FLAG", 135), ("NON_COMPLIANT_PROCEDURE", 133), ("INCORRECT_SERVICE_RECORD", 110)]
max_count = 217
bar_top = Inches(2.6)
for i, (name, count) in enumerate(anomalies):
    y = bar_top + Inches(i * 0.55)
    # Label
    lb = slide4.shapes.add_textbox(Inches(0.8), y, Inches(2.5), Inches(0.4))
    lbp = lb.text_frame.paragraphs[0]
    lbp.text = name
    lbp.font.size = Pt(13)
    lbp.font.color.rgb = WHITE
    lbp.alignment = PP_ALIGN.RIGHT
    # Bar
    bar_width = Inches(3.2 * count / max_count)
    bar = slide4.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(3.5), y + Inches(0.05), bar_width, Inches(0.3))
    bar.fill.solid()
    bar.fill.fore_color.rgb = ACCENT_ORANGE if i < 4 else SUBTITLE_BLUE
    bar.line.fill.background()
    # Count label
    cl = slide4.shapes.add_textbox(Inches(3.5) + bar_width + Inches(0.1), y, Inches(0.5), Inches(0.4))
    clp = cl.text_frame.paragraphs[0]
    clp.text = str(count)
    clp.font.size = Pt(13)
    clp.font.bold = True
    clp.font.color.rgb = WHITE

# Right side — Severity breakdown
txLabel2 = slide4.shapes.add_textbox(Inches(7.8), Inches(2.0), Inches(5), Inches(0.4))
tfL2 = txLabel2.text_frame
pL2 = tfL2.paragraphs[0]
pL2.text = "Severity Breakdown"
pL2.font.size = Pt(18)
pL2.font.bold = True
pL2.font.color.rgb = ACCENT_ORANGE

severities = [("CRITICAL", 128, CRITICAL_RED), ("HIGH", 264, HIGH_AMBER), ("MEDIUM", 346, MEDIUM_YELLOW), ("LOW", 231, LOW_GREEN)]
for i, (sev, cnt, clr) in enumerate(severities):
    bx = slide4.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(7.8), Inches(2.6 + i * 1.1), Inches(4.5), Inches(0.9))
    bx.fill.solid()
    bx.fill.fore_color.rgb = RGBColor(0x0F, 0x25, 0x3D)
    bx.line.color.rgb = clr
    bx.line.width = Pt(2)
    btf = bx.text_frame
    bp = btf.paragraphs[0]
    bp.alignment = PP_ALIGN.CENTER
    r1 = bp.add_run()
    r1.text = f"{sev}: "
    r1.font.size = Pt(22)
    r1.font.bold = True
    r1.font.color.rgb = clr
    r2 = bp.add_run()
    r2.text = f"{cnt} events"
    r2.font.size = Pt(22)
    r2.font.color.rgb = WHITE

add_bottom_bar(slide4)
add_slide_number(slide4, 4)


# ═══════════════════════════════════════════════════
# SLIDE 5 — ML-Powered Risk Assessment
# ═══════════════════════════════════════════════════
slide5 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide5)
add_section_title(slide5, "ML-Powered Failure Prediction", "Part failure probability & recommended action for every component")

# Risk distribution boxes
risk_cats = [
    ("CRITICAL", "280", CRITICAL_RED),
    ("HIGH", "257", HIGH_AMBER),
    ("MEDIUM", "464", MEDIUM_YELLOW),
    ("LOW", "639", LOW_GREEN),
]
for i, (cat, cnt, clr) in enumerate(risk_cats):
    bx = slide5.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.8 + i * 3.0), Inches(2.2), Inches(2.7), Inches(1.6))
    bx.fill.solid()
    bx.fill.fore_color.rgb = RGBColor(0x0F, 0x25, 0x3D)
    bx.line.color.rgb = clr
    bx.line.width = Pt(2.5)
    btf = bx.text_frame
    bp = btf.paragraphs[0]
    bp.alignment = PP_ALIGN.CENTER
    r1 = bp.add_run()
    r1.text = cnt
    r1.font.size = Pt(44)
    r1.font.bold = True
    r1.font.color.rgb = clr
    bp2 = btf.add_paragraph()
    bp2.alignment = PP_ALIGN.CENTER
    r2 = bp2.add_run()
    r2.text = cat
    r2.font.size = Pt(16)
    r2.font.color.rgb = RGBColor(0xBB, 0xBB, 0xBB)

bullets5 = [
    "GBT binary classifier (AUC 0.999) + Isolation Forest assigns failure probability to every Parker component",
    "5 recommended actions: ROUTINE_MONITORING → SCHEDULE_INSPECTION → PRIORITY_SERVICE → IMMEDIATE_REPLACEMENT → EMERGENCY_GROUND",
    "280 CRITICAL components identified — $3.3M warranty exposure in hydraulic actuators alone",
]
add_bullet_slide_content(slide5, bullets5, Inches(0.8), Inches(4.3), Inches(11.5), Inches(2.5), font_size=19)

add_bottom_bar(slide5)
add_slide_number(slide5, 5)


# ═══════════════════════════════════════════════════
# SLIDE 6 — Airline-Level Insights (Table)
# ═══════════════════════════════════════════════════
slide6 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide6)
add_section_title(slide6, "Customer Operations", "Compliance and warranty metrics across 10 airline customers")

airline_headers = ["Airline", "Fleet", "Compliance", "Anomalies", "Warranty $", "Top Anomaly"]
airline_data = [
    ["Continental Wings", "40", "81.0%", "132", "$890K", "MISSED_MAINT_INTERVAL"],
    ["TransOcean Air", "35", "81.5%", "109", "$779K", "PREMATURE_WEAR"],
    ["Meridian Airlines", "28", "81.9%", "82", "$717K", "MISSED_MAINT_INTERVAL"],
    ["Atlas Pacific", "38", "80.7%", "127", "$635K", "PREMATURE_WEAR"],
    ["SkyWest Global", "42", "82.8%", "117", "$618K", "MISSED_MAINT_INTERVAL"],
    ["Aurora Airlines", "36", "82.3%", "103", "$553K", "UNDOC_REPLACEMENT"],
    ["Northern Star Airways", "30", "82.3%", "86", "$481K", "UNDOC_REPLACEMENT"],
    ["Summit Aviation", "22", "81.1%", "70", "$457K", "MISSED_MAINT_INTERVAL"],
    ["Horizon Express", "32", "83.6%", "78", "$448K", "PREMATURE_WEAR"],
    ["Falcon Jet Airways", "25", "83.1%", "65", "$378K", "FALSE_COMPLIANCE"],
]

rows = len(airline_data) + 1
cols = len(airline_headers)
tbl = slide6.shapes.add_table(rows, cols, Inches(0.4), Inches(2.0), Inches(12.5), Inches(4.8)).table

# Column widths
col_widths = [Inches(2.5), Inches(0.9), Inches(1.3), Inches(1.2), Inches(1.3), Inches(2.8)]
for i, w in enumerate(col_widths):
    tbl.columns[i].width = w

# Header row
for j, h in enumerate(airline_headers):
    cell = tbl.cell(0, j)
    set_cell_text(cell, h, font_size=13, bold=True, color=WHITE, alignment=PP_ALIGN.CENTER)
    cell.fill.solid()
    cell.fill.fore_color.rgb = RGBColor(0x14, 0x2D, 0x4A)

# Data rows with alternating colors
for i, row_data in enumerate(airline_data):
    for j, val in enumerate(row_data):
        cell = tbl.cell(i + 1, j)
        fc = DARK_TEXT
        if j == 2:  # Compliance column — color by threshold
            rate = float(val.replace("%", ""))
            if rate < 81:
                fc = CRITICAL_RED
            elif rate < 82:
                fc = HIGH_AMBER
        set_cell_text(cell, val, font_size=12, color=fc, alignment=PP_ALIGN.CENTER if j > 0 else PP_ALIGN.LEFT)
        cell.fill.solid()
        cell.fill.fore_color.rgb = WHITE if i % 2 == 0 else LIGHT_GRAY

add_bottom_bar(slide6)
add_slide_number(slide6, 6)


# ═══════════════════════════════════════════════════
# SLIDE 7 — Critical Aircraft
# ═══════════════════════════════════════════════════
slide7 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide7)
add_section_title(slide7, "Urgent Action: Critical Components", "280 components flagged CRITICAL — immediate maintenance intervention required")

# Warning banner
warn = slide7.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.8), Inches(1.7), Inches(11.7), Inches(0.6))
warn.fill.solid()
warn.fill.fore_color.rgb = RGBColor(0x3D, 0x14, 0x14)
warn.line.color.rgb = CRITICAL_RED
warn.line.width = Pt(2)
wtf = warn.text_frame
wp = wtf.paragraphs[0]
wp.alignment = PP_ALIGN.CENTER
wr = wp.add_run()
wr.text = "\u26A0  CRITICAL ALERT: 280 Parker components at CRITICAL failure risk — $8.5M warranty exposure"
wr.font.size = Pt(18)
wr.font.bold = True
wr.font.color.rgb = CRITICAL_RED

crit_headers = ["Component ID", "Type", "Aircraft", "Airline", "ML Prob", "Days to Fail"]
crit_data = [
    ["PH-HA-N10162", "HYDRAULIC_ACTUATOR", "N10162", "Falcon Jet", "99.9%", "1 (URGENT!)"],
    ["PH-FCV-N10017", "FUEL_CONTROL_VALVE", "N10017", "SkyWest", "99.9%", "3"],
    ["PH-AVS-N10001", "AVIONICS_SENSOR", "N10001", "SkyWest", "99.9%", "4"],
    ["PH-PV-N10167", "PNEUMATIC_VALVE", "N10167", "TransOcean", "99.9%", "8"],
    ["PH-HA-N10307", "HYDRAULIC_ACT", "N10307", "Aurora", "99.9%", "21"],
]

tbl2 = slide7.shapes.add_table(len(crit_data) + 1, len(crit_headers), Inches(0.8), Inches(2.7), Inches(11.7), Inches(3.0)).table
crit_widths = [Inches(2.2), Inches(2.5), Inches(1.5), Inches(1.8), Inches(1.4), Inches(2.0)]
for i, w in enumerate(crit_widths):
    tbl2.columns[i].width = w

for j, h in enumerate(crit_headers):
    cell = tbl2.cell(0, j)
    set_cell_text(cell, h, font_size=14, bold=True, color=WHITE, alignment=PP_ALIGN.CENTER)
    cell.fill.solid()
    cell.fill.fore_color.rgb = RGBColor(0x5C, 0x1A, 0x1A)

for i, row_data in enumerate(crit_data):
    for j, val in enumerate(row_data):
        cell = tbl2.cell(i + 1, j)
        fc = DARK_TEXT
        if j == 4:  # ML Prob
            fc = CRITICAL_RED
        if j == 5 and "URGENT" in val:
            fc = CRITICAL_RED
        set_cell_text(cell, val, font_size=13, color=fc, alignment=PP_ALIGN.CENTER, bold=(j == 5 and "URGENT" in val))
        cell.fill.solid()
        cell.fill.fore_color.rgb = WHITE if i % 2 == 0 else LIGHT_GRAY

# Footnote
fn = slide7.shapes.add_textbox(Inches(0.8), Inches(6.0), Inches(11), Inches(0.5))
fnp = fn.text_frame.paragraphs[0]
fnp.text = "PH-HA-N10162 has only 1 day to predicted failure — $28,500 hydraulic actuator on Falcon Jet N10162 requires immediate grounding."
fnp.font.size = Pt(15)
fnp.font.italic = True
fnp.font.color.rgb = HIGH_AMBER

add_bottom_bar(slide7)
add_slide_number(slide7, 7)


# ═══════════════════════════════════════════════════
# SLIDE 8 — Business Value & ROI
# ═══════════════════════════════════════════════════
slide8 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide8)
add_section_title(slide8, "Business Value & ROI", "Turning data into operational excellence")

value_items = [
    ("\U0001F6E1\uFE0F", "Proactive Part Failure Prevention", "ML predicts component failure before it happens — 99.9% accuracy on critical parts"),
    ("\u2699\uFE0F", "Reduce Warranty Exposure", "$8.5M warranty liability identified and prioritized for proactive intervention"),
    ("\u26A0\uFE0F", "Maintenance Anomaly Detection", "6 anomaly types detected across 9,839 service records — avg 37 days undetected before"),
    ("\U0001F4CA", "Customer Operations Visibility", "1,640 components across 10 airlines — compliance scorecards and SLA breach alerts"),
    ("\u2B50", "Databricks Lakehouse + ML + Genie AI", "Unified platform: medallion pipeline, GBT + Isolation Forest, NL queries via Genie Room"),
]

for i, (icon, title, desc) in enumerate(value_items):
    y = Inches(2.1 + i * 0.95)
    # Box
    bx = slide8.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.8), y, Inches(11.7), Inches(0.8))
    bx.fill.solid()
    bx.fill.fore_color.rgb = RGBColor(0x0F, 0x25, 0x3D)
    bx.line.color.rgb = RGBColor(0x2A, 0x5A, 0x8C)
    bx.line.width = Pt(1)
    # Title
    ttx = slide8.shapes.add_textbox(Inches(1.0), y + Inches(0.05), Inches(4), Inches(0.4))
    tp = ttx.text_frame.paragraphs[0]
    tr = tp.add_run()
    tr.text = title
    tr.font.size = Pt(20)
    tr.font.bold = True
    tr.font.color.rgb = ACCENT_ORANGE
    # Description
    dtx = slide8.shapes.add_textbox(Inches(1.0), y + Inches(0.38), Inches(11), Inches(0.4))
    dp = dtx.text_frame.paragraphs[0]
    dr = dp.add_run()
    dr.text = desc
    dr.font.size = Pt(16)
    dr.font.color.rgb = RGBColor(0xBB, 0xBB, 0xBB)

add_bottom_bar(slide8)
add_slide_number(slide8, 8)


# ═══════════════════════════════════════════════════
# SLIDE 9 — Thank You / Q&A
# ═══════════════════════════════════════════════════
slide9 = prs.slides.add_slide(prs.slide_layouts[6])
add_background(slide9)

# Top accent
top9 = slide9.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, SLIDE_W, Inches(0.06))
top9.fill.solid()
top9.fill.fore_color.rgb = ACCENT_ORANGE
top9.line.fill.background()

# "Questions?" centered
txBox = slide9.shapes.add_textbox(Inches(0), Inches(2.2), SLIDE_W, Inches(1.5))
tf = txBox.text_frame
p = tf.paragraphs[0]
p.alignment = PP_ALIGN.CENTER
run = p.add_run()
run.text = "Questions?"
run.font.size = Pt(56)
run.font.bold = True
run.font.color.rgb = WHITE

add_accent_line(slide9, Inches(5.2), Inches(3.8), Inches(3), ACCENT_ORANGE, Inches(0.05))

txBox2 = slide9.shapes.add_textbox(Inches(0), Inches(4.2), SLIDE_W, Inches(0.8))
tf2 = txBox2.text_frame
p2 = tf2.paragraphs[0]
p2.alignment = PP_ALIGN.CENTER
run2 = p2.add_run()
run2.text = "Parker Hannifin Component Predictive Maintenance — Powered by Databricks"
run2.font.size = Pt(22)
run2.font.color.rgb = SUBTITLE_BLUE

# Footer
txBox3 = slide9.shapes.add_textbox(Inches(0), Inches(5.8), SLIDE_W, Inches(0.4))
tf3 = txBox3.text_frame
p3 = tf3.paragraphs[0]
p3.alignment = PP_ALIGN.CENTER
run3 = p3.add_run()
run3.text = "databricks.com"
run3.font.size = Pt(14)
run3.font.color.rgb = DBX_RED

add_bottom_bar(slide9)
add_slide_number(slide9, 9)


# ═══════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════
output_path = "/Workspace/Users/ayman.el-ghazali@databricks.com/Parker_Hannifin_Component_Maintenance_Presentation.pptx"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
prs.save(output_path)

print(f"\n{'='*60}")
print(f"  Presentation saved successfully!")
print(f"  Path: {output_path}")
print(f"  Slides: {len(prs.slides)}")
print(f"{'='*60}")


  Presentation saved successfully!
  Path: /Workspace/Users/ayman.el-ghazali@databricks.com/Aircraft_Predictive_Maintenance_Presentation.pptx
  Slides: 9
